# Exploratory Data Analysis (EDA) - Smart Inventory Forecasting
This notebook contains the EDA for the preprocessed inventory dataset, focusing on top-selling items, slow-moving items, sales trends, and stock risks.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data
We will load the aggregated timeseries dataset for trend analysis and a sample of the granular clean dataset for stock level analysis.

In [2]:
df_ts = pd.read_csv('../Dataset/processed/inventory_timeseries.csv')
df_ts['date'] = pd.to_datetime(df_ts['date'])

df_clean = pd.read_csv('../Dataset/processed/inventory_clean.csv', usecols=['product_name', 'current_stock', 'reorder_point', 'recommended_restock'])
df_ts.head()

## 2. Top-Selling Products (Revenue)
Identifying which products bring in the most revenue.

In [3]:
top_selling = df_ts.groupby('product_name')['sales_idr'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_selling.values, y=top_selling.index, palette="viridis")
plt.title('Top 10 Products by Total Sales (IDR)')
plt.xlabel('Total Sales (IDR)')
plt.ylabel('Product')
plt.show()

## 3. Slow-Moving Products (Quantity Sold)
Identifying products with the lowest sales volume to avoid overstocking.

In [4]:
slow_moving = df_ts.groupby('product_name')['quantity_sold'].sum().sort_values(ascending=True).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=slow_moving.values, y=slow_moving.index, palette="magma")
plt.title('Top 10 Slow-Moving Products by Quantity Sold')
plt.xlabel('Quantity Sold')
plt.ylabel('Product')
plt.show()

## 4. Sales Trends Over Time
Analyzing monthly sales trends to identify seasonality or growth.

In [5]:
daily_sales = df_ts.groupby('date')['sales_idr'].sum().reset_index()
monthly_sales = daily_sales.set_index('date').resample('ME').sum().reset_index()
plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly_sales, x='date', y='sales_idr', marker='o', color='b')
plt.title('Monthly Sales Trend (IDR)')
plt.xlabel('Date')
plt.ylabel('Total Sales (IDR)')
plt.xticks(rotation=45)
plt.show()

## 5. Stock Movement and Risk Analysis
Comparing average current stock with reorder points to identify items prone to understocking.

In [6]:
stock_analysis = df_clean.groupby('product_name').agg({
    'current_stock': 'mean',
    'reorder_point': 'mean',
    'recommended_restock': 'sum'
}).sort_values('recommended_restock', ascending=False).head(15)

plt.figure(figsize=(12, 6))
x = range(len(stock_analysis))
width = 0.35
plt.bar([i - width/2 for i in x], stock_analysis['current_stock'], width, label='Avg Current Stock', color='skyblue')
plt.bar([i + width/2 for i in x], stock_analysis['reorder_point'], width, label='Reorder Point', color='salmon')
plt.title('Stock Levels vs Reorder Point (Top 15 Most Restocked)')
plt.xlabel('Product')
plt.ylabel('Quantity')
plt.xticks(x, stock_analysis.index, rotation=45, ha='right')
plt.legend()
plt.show()